# Creating and Using Custom Validation Rules in PyTestLab

Use a declarative bench rule to reject an unsuitable configuration before accessing
an instrument. This example uses a built-in simulated power supply.

## 1. Define a bench validation rule

Set `custom_validations` to a list of restricted expressions. An expression can read
configured instrument fields by alias. Here the bench requires an explicit maximum
of no more than 5 V on channel 1.

In [ ]:
from copy import deepcopy

from pytestlab import Bench

config = {
    "bench_name": "Validation tutorial",
    "simulate": True,
    "instruments": {
        "psu": {
            "profile": "keysight/EDU36311A",
            "safety_limits": {"channels": {1: {"voltage": {"max": 5.0}}}},
        }
    },
    "custom_validations": ["0 < psu['safety_limits']['channels'][1]['voltage']['max'] <= 5"],
}

## 2. Open a valid simulated bench

Use a dictionary or the path to a YAML file with `Bench.open()`. Passing
`force_simulation=True` restricts the example to built-in simulation backends.
The custom rule checks configuration; channel safety limits are a separate
runtime check.

In [ ]:
with Bench.open(config, force_simulation=True, start_experiment=False) as bench:
    psu = bench.psu
    psu.set_voltage(1, 3.3)
    voltage = float(psu.read_voltage(1))
    assert voltage == 3.3
    print(f"Read back {voltage:g} V from the simulated supply.")

## 3. Handle an out-of-policy configuration

Changing the declared limit to 6 V fails the custom rule before the bench
initializes its instruments.

In [ ]:
invalid = deepcopy(config)
invalid["instruments"]["psu"]["safety_limits"]["channels"][1]["voltage"]["max"] = 6.0

try:
    with Bench.open(invalid, force_simulation=True, start_experiment=False):
        raise AssertionError("The invalid bench configuration was accepted")
except ValueError as error:
    assert "Custom validation failed" in str(error)
    print(f"Caught the expected validation error: {error}")

The valid and invalid configurations exercise the same bench rule without connecting
to hardware.